# Trace Explorer: Parametric Knowledge vs. Search Tradeoff

Interactive exploration of model reasoning traces across pathology types and trajectory classifications.

**Models:** Gemini 3 Pro (strong parametric) | Nemotron 3 Nano (weak parametric)

In [1]:
import json
import glob
import re
import textwrap
from pathlib import Path
from IPython.display import display, HTML, Markdown

import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 50)

## 1. Load Data

In [2]:
BASE = Path("../")
LOGS = BASE / "logs" / "entity_questions"
RESULTS = BASE / "results" / "entity_questions"

MODELS = {
    "gemini": {"name": "Gemini 3 Pro", "dir": "gemini_3_pro"},
    "nemotron": {"name": "Nemotron 3 Nano", "dir": "nemotron_3_nano"},
}


def extract_question(problem_text: str) -> str:
    """Extract the actual question from the full prompt."""
    marker = "Now answer the following:"
    if marker in problem_text:
        remainder = problem_text.split(marker, 1)[1]
    else:
        remainder = problem_text
    # Find Question: ... Answer:
    q_match = re.search(r"Question:\s*(.+?)\s*Answer:", remainder, re.DOTALL)
    if q_match:
        return q_match.group(1).strip()
    # Fallback: just return first 100 chars
    return remainder.strip()[:100]


def load_traces(model_key: str) -> dict:
    """Load all trace files for a model, keyed by (question, run_id)."""
    model_dir = LOGS / MODELS[model_key]["dir"]
    trace_files = sorted(glob.glob(str(model_dir / "baseline_agent_run_*_traces_*.json")))
    traces = {}
    for fpath in trace_files:
        m = re.search(r"run_(\d+)", fpath)
        run_id = int(m.group(1)) if m else 0
        with open(fpath) as f:
            data = json.load(f)
        for entry in data:
            question = extract_question(entry["problem"])
            traces[(question, run_id)] = entry
    print(f"Loaded {len(traces)} traces for {MODELS[model_key]['name']} from {len(trace_files)} files")
    return traces


def load_analysis(model_key: str) -> pd.DataFrame:
    """Load merged_data.csv for a model."""
    csv_path = RESULTS / MODELS[model_key]["dir"] / "merged_data.csv"
    df = pd.read_csv(csv_path)
    df["model"] = model_key
    # Filter to search runs only
    if "is_no_search" in df.columns:
        df = df[df["is_no_search"] != True].copy()
    return df

In [3]:
# Load everything
traces = {}
analysis = {}
for key in MODELS:
    traces[key] = load_traces(key)
    analysis[key] = load_analysis(key)

df_all = pd.concat(analysis.values(), ignore_index=True)
print(f"\nTotal analysis rows: {len(df_all)}")
print(f"Gemini: {len(analysis['gemini'])} rows, Nemotron: {len(analysis['nemotron'])} rows")

Loaded 1000 traces for Gemini 3 Pro from 5 files
Loaded 1000 traces for Nemotron 3 Nano from 5 files

Total analysis rows: 1850
Gemini: 890 rows, Nemotron: 960 rows


## 2. Trace Matching & Display Utilities

In [4]:
def match_trace(model_key: str, problem_id: str, run_id: int = 1):
    """Find the trace entry matching a problem_id from the CSV."""
    # problem_id in CSV ends with '...', try matching
    question_prefix = problem_id.rstrip(".")
    model_traces = traces[model_key]
    # Exact match first
    for (q, rid), entry in model_traces.items():
        if rid == run_id and (q == question_prefix or q.startswith(question_prefix) or question_prefix.startswith(q.rstrip('.'))):
            return entry
    # Fuzzy match - prefix
    prefix = question_prefix[:40]
    for (q, rid), entry in model_traces.items():
        if rid == run_id and q.startswith(prefix):
            return entry
    return None


def _escape_html(text: str) -> str:
    """Escape HTML special characters."""
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def format_part(part: dict) -> str:
    """Format a single message part for display."""
    ptype = part.get("type", "unknown")

    if ptype == "thinking":
        content = part.get("content", "")
        text = _escape_html(content if len(content) <= 2000 else content[:2000] + "\n... [truncated]")
        return (
            '<div style="background:#f0f0f0; border-left:3px solid #888; padding:8px; margin:4px 0; font-size:13px;">'
            f'<b>🧠 Thinking:</b><br><pre style="white-space:pre-wrap; font-size:12px;">{text}</pre></div>'
        )

    elif ptype == "tool_call":
        tool_name = part.get("tool_name", "unknown")
        # Field is "arguments" in traces (may be str or dict)
        args = part.get("arguments", part.get("args", {}))
        if isinstance(args, str):
            try:
                args = json.loads(args)
            except (json.JSONDecodeError, TypeError):
                pass
        if tool_name == "search":
            query = args.get("query", str(args)) if isinstance(args, dict) else str(args)
            return (
                '<div style="background:#e8f4fd; border-left:3px solid #2196F3; padding:8px; margin:4px 0;">'
                f'<b>🔍 Search:</b> <code>{_escape_html(query)}</code></div>'
            )
        elif tool_name == "final_result":
            answer = args.get("answer", "") if isinstance(args, dict) else str(args)
            explanation = args.get("explanation", "") if isinstance(args, dict) else ""
            return (
                '<div style="background:#e8f5e9; border-left:3px solid #4CAF50; padding:8px; margin:4px 0;">'
                f'<b>✅ Final Answer:</b> <code>{_escape_html(str(answer))}</code>'
                f'<br><small>{_escape_html(str(explanation)[:500])}</small></div>'
            )
        else:
            args_str = json.dumps(args, indent=2)[:500] if isinstance(args, dict) else str(args)[:500]
            return (
                f'<div style="background:#fff3e0; border-left:3px solid #FF9800; padding:8px; margin:4px 0;">'
                f'<b>🔧 Tool: {tool_name}</b><br>'
                f'<pre style="font-size:11px;">{_escape_html(args_str)}</pre></div>'
            )

    elif ptype == "tool_call_response":
        # Field is "result" in traces (not "content")
        result = part.get("result", part.get("content", ""))
        tool_name = part.get("tool_name", "")

        # Parse if string
        if isinstance(result, str):
            try:
                parsed = json.loads(result)
            except (json.JSONDecodeError, TypeError):
                parsed = result
        else:
            parsed = result

        # Search results: list of {title, snippet, ...}
        if isinstance(parsed, list) and len(parsed) > 0 and isinstance(parsed[0], dict) and "title" in parsed[0]:
            snippets = []
            for r in parsed[:5]:
                title = _escape_html(r.get("title", ""))
                snippet = _escape_html(r.get("snippet", "")[:200])
                snippets.append(f"<li><b>{title}</b>: {snippet}</li>")
            more = f"<li><i>... and {len(parsed) - 5} more results</i></li>" if len(parsed) > 5 else ""
            items = "\n".join(snippets) + more
            return (
                '<div style="background:#f3e5f5; border-left:3px solid #9C27B0; padding:8px; margin:4px 0; font-size:12px;">'
                f'<b>📄 Search Results ({len(parsed)} total):</b><ul>{items}</ul></div>'
            )
        else:
            text = _escape_html(str(parsed)[:300])
            return f'<div style="background:#f5f5f5; padding:4px; margin:2px 0; font-size:11px;"><i>Tool response:</i> {text}</div>'

    elif ptype == "text":
        content = part.get("content", "")
        if len(content.strip()) == 0:
            return ""
        text = _escape_html(content if len(content) <= 1500 else content[:1500] + "\n... [truncated]")
        return f'<div style="padding:4px; margin:2px 0;"><pre style="white-space:pre-wrap; font-size:12px;">{text}</pre></div>'

    return f"<div><i>[{ptype}]</i> {_escape_html(str(part)[:200])}</div>"


ROLE_STYLES = {
    "system": ("⚙️ System", "#f5f5f5", "#999"),
    "user": ("👤 User", "#fff8e1", "#f57f17"),
    "assistant": ("🤖 Assistant", "#e3f2fd", "#1565c0"),
}


def render_trace(entry: dict, show_system: bool = False, show_user_prompt: bool = False):
    """Render a full trace as HTML."""
    question = extract_question(entry["problem"])
    meta = entry.get("metadata", {})

    html_parts = ['<div style="border:2px solid #333; border-radius:8px; padding:12px; margin:8px 0; max-width:900px;">']
    html_parts.append(f'<h3 style="margin:0 0 8px 0;">📝 {_escape_html(question)}</h3>')
    html_parts.append(
        f'<div style="font-size:11px; color:#666; margin-bottom:8px;">'
        f'Messages: {meta.get("total_messages", "?")} | '
        f'Tool calls: {meta.get("tool_calls", "?")} | '
        f'Thinking: {meta.get("thinking_messages", "?")}</div>'
    )

    for msg in entry.get("message_trace", []):
        role = msg.get("role", "unknown")
        if role == "system" and not show_system:
            continue

        label, bg, border = ROLE_STYLES.get(role, (role, "#fff", "#000"))

        # Skip the user prompt (usually long with examples) unless requested
        parts = msg.get("parts", [])
        if role == "user" and not show_user_prompt:
            has_tool_response = any(p.get("type") == "tool_call_response" for p in parts)
            if not has_tool_response:
                continue

        parts_html = [format_part(p) for p in parts]
        parts_html = [p for p in parts_html if p]
        if not parts_html:
            continue

        html_parts.append(
            f'<div style="border-left:4px solid {border}; background:{bg}; '
            f'padding:8px; margin:6px 0; border-radius:4px;">'
            f'<b>{label}</b>'
            + "\n".join(parts_html) + '</div>'
        )

    html_parts.append("</div>")
    display(HTML("\n".join(html_parts)))

## 3. Browse by Category

Helper to select and display traces filtered by analysis attributes.

In [5]:
def browse(
    model: str = "gemini",
    trajectory: str | None = None,
    pathology: str | None = None,
    epistemic_state: str | None = None,
    correct: bool | None = None,
    run_id: int = 1,
    max_display: int = 3,
    show_summary: bool = True,
) -> pd.DataFrame:
    """
    Filter analysis rows and display matching traces.

    Parameters
    ----------
    model : 'gemini' or 'nemotron'
    trajectory : e.g. 'STUBBORN_REJECTION', 'MISLED_BY_SEARCH', 'VERIFY_AND_CONFIRM', etc.
    pathology : column name, e.g. 'is_context_poisoning', 'is_performative_ignorance',
                'is_utilization_failure', 'is_confirmation_bias'
    epistemic_state : 'SEARCH_FIRST', 'IGNORANCE', 'AMBIGUITY', 'HIGH_CERTAINTY'
    correct : True/False to filter by agent_correct
    run_id : which run to show traces from (1-5)
    max_display : max number of traces to render
    show_summary : whether to show the analysis summary table

    Returns filtered DataFrame.
    """
    df = analysis[model].copy()
    df = df[df["run_id"] == run_id]

    if trajectory:
        df = df[df["trajectory_type"] == trajectory]
    if pathology:
        df = df[df[pathology] == True]
    if epistemic_state:
        df = df[df["epistemic_state"] == epistemic_state]
    if correct is not None:
        df = df[df["agent_correct"] == correct]

    label = MODELS[model]["name"]
    filters = []
    if trajectory: filters.append(f"trajectory={trajectory}")
    if pathology: filters.append(f"{pathology}=True")
    if epistemic_state: filters.append(f"state={epistemic_state}")
    if correct is not None: filters.append(f"correct={correct}")
    filter_str = ", ".join(filters) if filters else "no filters"

    display(Markdown(f"### {label} — {filter_str}\n**{len(df)} matching rows** (showing up to {max_display} traces from run {run_id})"))

    if show_summary and len(df) > 0:
        summary_cols = ["problem_id", "agent_correct", "epistemic_state", "trajectory_type",
                        "num_searches", "entropy", "hypothesis_correct"]
        summary_cols = [c for c in summary_cols if c in df.columns]
        display(df[summary_cols].head(10))

    for i, (_, row) in enumerate(df.head(max_display).iterrows()):
        entry = match_trace(model, row["problem_id"], run_id)
        if entry:
            # Show analysis context
            ctx_parts = []
            for col in ["epistemic_state", "trajectory_type", "hypothesis_correct",
                        "pre_search_state", "round_1_state", "round_1_evidence_eval"]:
                if col in row and pd.notna(row[col]):
                    ctx_parts.append(f"<b>{col}:</b> {row[col]}")
            # Show pathology flags
            for flag in ["is_context_poisoning", "is_performative_ignorance",
                         "is_utilization_failure", "is_confirmation_bias"]:
                if flag in row and row[flag]:
                    ctx_parts.append(f'<span style="color:red;"><b>{flag}</b></span>')
            # Show round summaries
            for r in range(1, 5):
                scol = f"round_{r}_summary"
                if scol in row and pd.notna(row[scol]):
                    ctx_parts.append(f"<b>Round {r} summary:</b> <i>{str(row[scol])[:300]}</i>")

            correct_str = '✅' if row.get('agent_correct') else '❌'
            display(HTML(
                f'<div style="background:#fffde7; border:1px solid #fdd835; padding:8px; '
                f'margin:8px 0; border-radius:4px; font-size:12px;">'
                f'<b>Analysis Context</b> {correct_str} (entropy={row.get("entropy", "?")}, '
                f'searches={row.get("num_searches", "?")})'
                f'<br>{"<br>".join(ctx_parts)}</div>'
            ))
            render_trace(entry)
        else:
            display(HTML(f'<div style="color:red;">⚠️ No trace found for: {row["problem_id"]}</div>'))

    return df

## 4. Distribution Overview

See what's available to explore.

In [6]:
for key in MODELS:
    df = analysis[key][analysis[key]["run_id"] == 1]
    display(Markdown(f"### {MODELS[key]['name']} (run 1, n={len(df)})"))

    print("Trajectory types:")
    print(df["trajectory_type"].value_counts().to_string())
    print()
    print("Epistemic states:")
    print(df["epistemic_state"].value_counts().to_string())
    print()

    flags = ["is_context_poisoning", "is_performative_ignorance",
             "is_utilization_failure", "is_confirmation_bias"]
    flag_counts = {f: int(df[f].sum()) for f in flags if f in df.columns}
    print("Pathology flags:")
    for f, c in flag_counts.items():
        print(f"  {f}: {c}")
    print("---")

### Gemini 3 Pro (run 1, n=178)

Trajectory types:
trajectory_type
VERIFY_AND_CONFIRM    54
EXPLORE_AND_FIND      43
HEDGE_AND_RESOLVE     27
PRODUCTIVE_SEARCH     12
NEUTRAL_TRAJECTORY    12
STUBBORN_REJECTION     5
MISLED_BY_SEARCH       4
EXPLORE_AND_FAIL       1

Epistemic states:
epistemic_state
HIGH_CERTAINTY    86
AMBIGUITY         37
SEARCH_FIRST      34
IGNORANCE         21

Pathology flags:
  is_context_poisoning: 12
  is_performative_ignorance: 51
  is_utilization_failure: 24
  is_confirmation_bias: 0
---


### Nemotron 3 Nano (run 1, n=192)

Trajectory types:
trajectory_type
EXPLORE_AND_FIND      66
PRODUCTIVE_SEARCH     36
HEDGE_AND_RESOLVE     33
NEUTRAL_TRAJECTORY    31
VERIFY_AND_CONFIRM     7
MISLED_BY_SEARCH       3
VERIFY_AND_CORRECT     1

Epistemic states:
epistemic_state
AMBIGUITY         105
IGNORANCE          37
SEARCH_FIRST       29
HIGH_CERTAINTY     21

Pathology flags:
  is_context_poisoning: 1
  is_performative_ignorance: 14
  is_utilization_failure: 21
  is_confirmation_bias: 0
---


---
## 5. Explore Specific Pathologies & Trajectories

Use `browse()` below to explore any combination. Examples are provided for the key phenomena.

### 5.1 STUBBORN_REJECTION — Model ignores contradicting search evidence

In [7]:
_ = browse(model="gemini", trajectory="STUBBORN_REJECTION", max_display=2)

### Gemini 3 Pro — trajectory=STUBBORN_REJECTION
**5 matching rows** (showing up to 2 traces from run 1)

,problem_id,agent_correct,epistemic_state,trajectory_type,num_searches,entropy,hypothesis_correct
2,Who is the author of The Forest?...,True,HIGH_CERTAINTY,STUBBORN_REJECTION,1.0,1.5219,NO
27,Who is the author of Limbo?...,False,HIGH_CERTAINTY,STUBBORN_REJECTION,2.0,0.7219,NO
55,What music label is Skin represented by?...,False,HIGH_CERTAINTY,STUBBORN_REJECTION,7.0,0.0000,NO
84,What music label is Good Morning Beautiful represe...,False,HIGH_CERTAINTY,STUBBORN_REJECTION,5.0,0.0000,NO
92,What music label is Leif Garrett represented by?...,True,HIGH_CERTAINTY,STUBBORN_REJECTION,1.0,0.9710,NO


### 5.2 MISLED_BY_SEARCH — Search flips model from correct to incorrect

In [8]:
_ = browse(model="gemini", trajectory="MISLED_BY_SEARCH", max_display=2)

### Gemini 3 Pro — trajectory=MISLED_BY_SEARCH
**4 matching rows** (showing up to 2 traces from run 1)

,problem_id,agent_correct,epistemic_state,trajectory_type,num_searches,entropy,hypothesis_correct
52,What music label is I Could Fall in Love represent...,False,HIGH_CERTAINTY,MISLED_BY_SEARCH,1.0,0.000,YES
113,Who is James Connolly married to?...,False,HIGH_CERTAINTY,MISLED_BY_SEARCH,1.0,0.000,YES
163,Which company is Macchi M.12 produced by?...,False,HIGH_CERTAINTY,MISLED_BY_SEARCH,1.0,0.971,YES
182,Which company is Dealer Team Vauxhall produced by?...,False,HIGH_CERTAINTY,MISLED_BY_SEARCH,1.0,0.000,YES


In [9]:
_ = browse(model="nemotron", trajectory="MISLED_BY_SEARCH", max_display=2)

### Nemotron 3 Nano — trajectory=MISLED_BY_SEARCH
**3 matching rows** (showing up to 2 traces from run 1)

,problem_id,agent_correct,epistemic_state,trajectory_type,num_searches,entropy,hypothesis_correct
28,Who is the author of The Lord of the Isles?...,False,AMBIGUITY,MISLED_BY_SEARCH,1.0,1.9219,YES
35,Who is the author of Introduction to Christianity?...,False,AMBIGUITY,MISLED_BY_SEARCH,1.0,1.5219,YES
164,Which company is Daewoo Tosca produced by?...,False,AMBIGUITY,MISLED_BY_SEARCH,1.0,0.0000,YES


### 5.3 Context Poisoning — Search hurts a correct parametric answer

In [10]:
_ = browse(model="gemini", pathology="is_context_poisoning", max_display=2)

### Gemini 3 Pro — is_context_poisoning=True
**12 matching rows** (showing up to 2 traces from run 1)

,problem_id,agent_correct,epistemic_state,trajectory_type,num_searches,entropy,hypothesis_correct
18,Who is the author of The Ghosts of London?...,False,AMBIGUITY,HEDGE_AND_RESOLVE,3.0,0.7219,YES
35,Who is the author of Introduction to Christianity?...,False,HIGH_CERTAINTY,VERIFY_AND_CONFIRM,1.0,0.0000,YES
101,Who is Julie Peasgood married to?...,False,SEARCH_FIRST,PRODUCTIVE_SEARCH,3.0,0.0000,NaN
104,Who is Fatimah married to?...,False,HIGH_CERTAINTY,VERIFY_AND_CONFIRM,1.0,0.0000,YES
117,Who is Mary Ritter Beard married to?...,False,HIGH_CERTAINTY,NaN,0.6,0.0000,YES
140,Who is Princess Caroline of Denmark married to?...,False,AMBIGUITY,HEDGE_AND_RESOLVE,1.0,0.0000,YES
156,Which company is Estonian icebreaker Suur Tõll pro...,False,AMBIGUITY,HEDGE_AND_RESOLVE,1.0,0.0000,YES
159,Which company is USS Joseph E. Campbell produced b...,False,IGNORANCE,EXPLORE_AND_FIND,1.0,0.0000,NaN
167,Which company is AR-15 produced by?...,False,HIGH_CERTAINTY,VERIFY_AND_CONFIRM,2.0,0.0000,YES
176,Which company is USS Wake Island produced by?...,False,HIGH_CERTAINTY,VERIFY_AND_CONFIRM,1.0,0.0000,YES


### 5.4 Performative Ignorance — Model claims not to know, but knows the answer

In [11]:
_ = browse(model="gemini", pathology="is_performative_ignorance", max_display=2)

### Gemini 3 Pro — is_performative_ignorance=True
**51 matching rows** (showing up to 2 traces from run 1)

,problem_id,agent_correct,epistemic_state,trajectory_type,num_searches,entropy,hypothesis_correct
0,Who is the author of File Under Popular?...,True,SEARCH_FIRST,EXPLORE_AND_FIND,1.0,0.0000,NaN
1,Who is the author of Showboat World?...,True,SEARCH_FIRST,EXPLORE_AND_FIND,1.0,0.0000,NaN
4,Who is the author of Summer of the Swans?...,True,SEARCH_FIRST,PRODUCTIVE_SEARCH,1.0,0.0000,NaN
10,Who is the author of The Overlook?...,True,SEARCH_FIRST,EXPLORE_AND_FIND,2.0,0.0000,NaN
11,Who is the author of Indian Summer?...,True,AMBIGUITY,NEUTRAL_TRAJECTORY,1.0,0.7219,YES
18,Who is the author of The Ghosts of London?...,False,AMBIGUITY,HEDGE_AND_RESOLVE,3.0,0.7219,YES
19,Who is the author of The Hundred-Year Christmas?...,True,SEARCH_FIRST,EXPLORE_AND_FIND,1.0,0.0000,NaN
29,Who is the author of Toad Rage?...,True,SEARCH_FIRST,EXPLORE_AND_FIND,1.0,0.0000,NaN
32,Who is the author of Hate on Trial?...,True,IGNORANCE,PRODUCTIVE_SEARCH,1.0,0.0000,NaN
34,Who is the author of The Asylum Seeker?...,True,SEARCH_FIRST,EXPLORE_AND_FIND,8.0,0.0000,NaN


In [12]:
# Compare: Nemotron's "ignorance" is genuine — it truly doesn't know
_ = browse(model="nemotron", epistemic_state="IGNORANCE", max_display=2)

### Nemotron 3 Nano — state=IGNORANCE
**37 matching rows** (showing up to 2 traces from run 1)

,problem_id,agent_correct,epistemic_state,trajectory_type,num_searches,entropy,hypothesis_correct
0,Who is the author of File Under Popular?...,True,IGNORANCE,EXPLORE_AND_FIND,1.0,2.3219,NaN
1,Who is the author of Showboat World?...,True,IGNORANCE,EXPLORE_AND_FIND,1.0,2.3219,NaN
11,Who is the author of Indian Summer?...,True,IGNORANCE,EXPLORE_AND_FIND,2.0,2.3219,NaN
23,Who is the author of The Witching Hour?...,True,IGNORANCE,EXPLORE_AND_FIND,1.0,0.0000,NaN
26,Who is the author of Kosala?...,True,IGNORANCE,EXPLORE_AND_FIND,1.0,1.5219,NaN
27,Who is the author of Limbo?...,False,IGNORANCE,EXPLORE_AND_FIND,2.0,1.9219,NaN
30,Who is the author of Miracle in the Rain?...,True,IGNORANCE,EXPLORE_AND_FIND,1.0,2.3219,NaN
32,Who is the author of Hate on Trial?...,True,IGNORANCE,EXPLORE_AND_FIND,1.0,1.9219,NaN
34,Who is the author of The Asylum Seeker?...,True,IGNORANCE,EXPLORE_AND_FIND,1.0,2.3219,NaN
41,Who is the author of The Seven Fabulous Wonders?...,True,IGNORANCE,EXPLORE_AND_FIND,1.0,2.3219,NaN


### 5.5 Utilization Failure — Search finds the answer, model doesn't use it

In [13]:
_ = browse(model="gemini", pathology="is_utilization_failure", max_display=2)

### Gemini 3 Pro — is_utilization_failure=True
**24 matching rows** (showing up to 2 traces from run 1)

,problem_id,agent_correct,epistemic_state,trajectory_type,num_searches,entropy,hypothesis_correct
18,Who is the author of The Ghosts of London?...,False,AMBIGUITY,HEDGE_AND_RESOLVE,3.0,0.7219,YES
35,Who is the author of Introduction to Christianity?...,False,HIGH_CERTAINTY,VERIFY_AND_CONFIRM,1.0,0.0000,YES
52,What music label is I Could Fall in Love represent...,False,HIGH_CERTAINTY,MISLED_BY_SEARCH,1.0,0.0000,YES
53,What music label is Píntame De Colores represented...,False,AMBIGUITY,HEDGE_AND_RESOLVE,2.0,1.9219,NO
63,What music label is Four Compositions (Quartet) 19...,False,SEARCH_FIRST,NEUTRAL_TRAJECTORY,1.0,0.0000,NaN
80,What music label is Some People Change represented...,False,IGNORANCE,NEUTRAL_TRAJECTORY,1.0,0.0000,NaN
95,What music label is Bandage represented by?...,False,AMBIGUITY,HEDGE_AND_RESOLVE,4.0,0.7219,YES
104,Who is Fatimah married to?...,False,HIGH_CERTAINTY,VERIFY_AND_CONFIRM,1.0,0.0000,YES
109,Who is Ahn Jae-hyun married to?...,False,AMBIGUITY,HEDGE_AND_RESOLVE,1.0,0.7219,NO
113,Who is James Connolly married to?...,False,HIGH_CERTAINTY,MISLED_BY_SEARCH,1.0,0.0000,YES


In [14]:
_ = browse(model="nemotron", pathology="is_utilization_failure", max_display=2)

### Nemotron 3 Nano — is_utilization_failure=True
**21 matching rows** (showing up to 2 traces from run 1)

,problem_id,agent_correct,epistemic_state,trajectory_type,num_searches,entropy,hypothesis_correct
35,Who is the author of Introduction to Christianity?...,False,AMBIGUITY,MISLED_BY_SEARCH,1.0,1.5219,YES
67,What music label is Mike Dred represented by?...,False,SEARCH_FIRST,EXPLORE_AND_FIND,1.0,2.3219,NaN
81,What music label is My Guitar Wants to Kill Your M...,False,AMBIGUITY,NEUTRAL_TRAJECTORY,1.0,1.9219,NO
86,What music label is Vince Staples represented by?...,False,AMBIGUITY,NEUTRAL_TRAJECTORY,3.0,0.9710,NO
87,What music label is James Taylor represented by?...,False,SEARCH_FIRST,EXPLORE_AND_FIND,1.0,1.3710,NaN
95,What music label is Bandage represented by?...,False,AMBIGUITY,NEUTRAL_TRAJECTORY,1.0,0.7219,NO
107,Who is Dagobert I married to?...,False,AMBIGUITY,NEUTRAL_TRAJECTORY,3.0,2.3219,NO
108,Who is Gloria Grahame married to?...,False,SEARCH_FIRST,EXPLORE_AND_FIND,1.0,2.3219,NaN
112,Who is Sahib Jamal married to?...,False,AMBIGUITY,NEUTRAL_TRAJECTORY,1.0,0.0000,NO
113,Who is James Connolly married to?...,False,SEARCH_FIRST,EXPLORE_AND_FIND,1.0,1.3710,NaN


### 5.6 VERIFY_AND_CONFIRM — Search confirms what the model already knew

In [15]:
_ = browse(model="gemini", trajectory="VERIFY_AND_CONFIRM", epistemic_state="HIGH_CERTAINTY", max_display=2)

### Gemini 3 Pro — trajectory=VERIFY_AND_CONFIRM, state=HIGH_CERTAINTY
**54 matching rows** (showing up to 2 traces from run 1)

,problem_id,agent_correct,epistemic_state,trajectory_type,num_searches,entropy,hypothesis_correct
3,Who is the author of E for Ecstasy?...,True,HIGH_CERTAINTY,VERIFY_AND_CONFIRM,1.0,0.0000,YES
15,Who is the author of Deploying Renewables 2011?...,True,HIGH_CERTAINTY,VERIFY_AND_CONFIRM,1.0,0.0000,YES
20,Who is the author of Daemon?...,True,HIGH_CERTAINTY,VERIFY_AND_CONFIRM,1.0,0.0000,YES
24,Who is the author of The Parasites?...,True,HIGH_CERTAINTY,VERIFY_AND_CONFIRM,1.0,0.0000,YES
33,Who is the author of Maximum Ride: Saving the Worl...,True,HIGH_CERTAINTY,VERIFY_AND_CONFIRM,1.0,0.0000,YES
35,Who is the author of Introduction to Christianity?...,False,HIGH_CERTAINTY,VERIFY_AND_CONFIRM,1.0,0.0000,YES
36,Who is the author of In Our Time?...,True,HIGH_CERTAINTY,VERIFY_AND_CONFIRM,3.0,0.0000,YES
45,Who is the author of In Odd We Trust?...,True,HIGH_CERTAINTY,VERIFY_AND_CONFIRM,1.0,0.7219,YES
48,Who is the author of Old New York?...,True,HIGH_CERTAINTY,VERIFY_AND_CONFIRM,1.0,0.0000,YES
54,What music label is Guy-Manuel de Homem-Christo re...,True,HIGH_CERTAINTY,VERIFY_AND_CONFIRM,1.0,2.3219,YES


### 5.7 PRODUCTIVE_SEARCH — Model genuinely learns from search

In [16]:
_ = browse(model="nemotron", trajectory="PRODUCTIVE_SEARCH", max_display=2)

### Nemotron 3 Nano — trajectory=PRODUCTIVE_SEARCH
**36 matching rows** (showing up to 2 traces from run 1)

,problem_id,agent_correct,epistemic_state,trajectory_type,num_searches,entropy,hypothesis_correct
4,Who is the author of Summer of the Swans?...,True,AMBIGUITY,PRODUCTIVE_SEARCH,1.0,0.7219,NO
6,Who is the author of The Midwife's Apprentice?...,True,AMBIGUITY,PRODUCTIVE_SEARCH,1.0,0.0000,NO
12,Who is the author of Plena Ilustrita Vortaro de Es...,True,AMBIGUITY,PRODUCTIVE_SEARCH,1.0,1.9219,NO
13,Who is the author of Profiles in Folly?...,True,AMBIGUITY,PRODUCTIVE_SEARCH,1.0,2.3219,NO
14,"Who is the author of It's OK, I'm Wearing Really B...",True,AMBIGUITY,PRODUCTIVE_SEARCH,1.0,2.3219,NO
18,Who is the author of The Ghosts of London?...,True,AMBIGUITY,PRODUCTIVE_SEARCH,1.0,2.3219,NO
22,Who is the author of Tom Cruise: All the World's a...,True,AMBIGUITY,PRODUCTIVE_SEARCH,1.0,0.7219,NO
29,Who is the author of Toad Rage?...,True,AMBIGUITY,PRODUCTIVE_SEARCH,1.0,1.9219,NaN
36,Who is the author of In Our Time?...,True,AMBIGUITY,PRODUCTIVE_SEARCH,1.0,0.7219,NO
48,Who is the author of Old New York?...,True,AMBIGUITY,PRODUCTIVE_SEARCH,1.0,1.9219,NaN


### 5.8 NEUTRAL_TRAJECTORY — Search has no clear effect (litmus test for parametric knowledge)

In [17]:
# Gemini with neutral trajectory — still often correct (parametric fallback)
_ = browse(model="gemini", trajectory="NEUTRAL_TRAJECTORY", correct=True, max_display=2)

### Gemini 3 Pro — trajectory=NEUTRAL_TRAJECTORY, correct=True
**8 matching rows** (showing up to 2 traces from run 1)

,problem_id,agent_correct,epistemic_state,trajectory_type,num_searches,entropy,hypothesis_correct
11,Who is the author of Indian Summer?...,True,AMBIGUITY,NEUTRAL_TRAJECTORY,1.0,0.7219,YES
46,Who is the author of Long Time Dead?...,True,AMBIGUITY,NEUTRAL_TRAJECTORY,7.0,2.3219,YES
57,What music label is Chris Stamey represented by?...,True,AMBIGUITY,NEUTRAL_TRAJECTORY,2.0,2.3219,YES
72,What music label is Good Hit represented by?...,True,HIGH_CERTAINTY,NEUTRAL_TRAJECTORY,2.0,0.0000,YES
77,What music label is Hawthorne Heights represented ...,True,HIGH_CERTAINTY,NEUTRAL_TRAJECTORY,2.0,0.0000,YES
86,What music label is Vince Staples represented by?...,True,AMBIGUITY,NEUTRAL_TRAJECTORY,3.0,0.0000,YES
91,What music label is The Seven Year Itch represente...,True,AMBIGUITY,NEUTRAL_TRAJECTORY,6.0,1.5219,YES
93,What music label is Significant Other represented ...,True,HIGH_CERTAINTY,NEUTRAL_TRAJECTORY,7.0,0.7219,YES


In [18]:
# Nemotron with neutral trajectory — often wrong (no parametric fallback)
_ = browse(model="nemotron", trajectory="NEUTRAL_TRAJECTORY", correct=False, max_display=2)

### Nemotron 3 Nano — trajectory=NEUTRAL_TRAJECTORY, correct=False
**16 matching rows** (showing up to 2 traces from run 1)

,problem_id,agent_correct,epistemic_state,trajectory_type,num_searches,entropy,hypothesis_correct
2,Who is the author of The Forest?...,False,AMBIGUITY,NEUTRAL_TRAJECTORY,1.0,2.3219,NO
16,Who is the author of Bedlam?...,False,AMBIGUITY,NEUTRAL_TRAJECTORY,1.0,1.9219,NO
62,What music label is Solace represented by?...,False,AMBIGUITY,NEUTRAL_TRAJECTORY,1.0,1.3710,NO
76,What music label is Sink or Swim represented by?...,False,AMBIGUITY,NEUTRAL_TRAJECTORY,1.0,1.5219,NO
81,What music label is My Guitar Wants to Kill Your M...,False,AMBIGUITY,NEUTRAL_TRAJECTORY,1.0,1.9219,NO
83,What music label is Player Piano represented by?...,False,AMBIGUITY,NEUTRAL_TRAJECTORY,2.0,0.0000,NO
84,What music label is Good Morning Beautiful represe...,False,AMBIGUITY,NEUTRAL_TRAJECTORY,1.0,1.9219,NO
86,What music label is Vince Staples represented by?...,False,AMBIGUITY,NEUTRAL_TRAJECTORY,3.0,0.9710,NO
95,What music label is Bandage represented by?...,False,AMBIGUITY,NEUTRAL_TRAJECTORY,1.0,0.7219,NO
107,Who is Dagobert I married to?...,False,AMBIGUITY,NEUTRAL_TRAJECTORY,3.0,2.3219,NO


### 5.9 HEDGE_AND_RESOLVE — Model was uncertain, search partially resolves

In [19]:
_ = browse(model="gemini", trajectory="HEDGE_AND_RESOLVE", max_display=2)

### Gemini 3 Pro — trajectory=HEDGE_AND_RESOLVE
**27 matching rows** (showing up to 2 traces from run 1)

,problem_id,agent_correct,epistemic_state,trajectory_type,num_searches,entropy,hypothesis_correct
16,Who is the author of Bedlam?...,True,AMBIGUITY,HEDGE_AND_RESOLVE,2.0,1.9219,YES
18,Who is the author of The Ghosts of London?...,False,AMBIGUITY,HEDGE_AND_RESOLVE,3.0,0.7219,YES
41,Who is the author of The Seven Fabulous Wonders?...,True,AMBIGUITY,HEDGE_AND_RESOLVE,1.0,0.7219,NO
50,What music label is Judgement Days represented by?...,True,AMBIGUITY,HEDGE_AND_RESOLVE,2.0,0.0000,YES
53,What music label is Píntame De Colores represented...,False,AMBIGUITY,HEDGE_AND_RESOLVE,2.0,1.9219,NO
59,What music label is The Age of Fear represented by...,True,AMBIGUITY,HEDGE_AND_RESOLVE,8.0,2.3219,YES
62,What music label is Solace represented by?...,False,AMBIGUITY,HEDGE_AND_RESOLVE,3.0,1.3710,NO
64,What music label is Ghost Dance represented by?...,False,AMBIGUITY,HEDGE_AND_RESOLVE,2.0,0.0000,NO
66,What music label is E=MC² represented by?...,True,AMBIGUITY,HEDGE_AND_RESOLVE,9.0,0.0000,YES
76,What music label is Sink or Swim represented by?...,False,AMBIGUITY,HEDGE_AND_RESOLVE,5.0,0.0000,YES


---
## 6. Side-by-Side Model Comparison

Compare how the two models handle the **same question**.

In [20]:
def compare_models(problem_prefix: str, run_id: int = 1):
    """
    Show traces for the same problem from both models side by side.

    problem_prefix: beginning of the question text (enough to uniquely match)
    """
    for model_key in MODELS:
        df = analysis[model_key]
        match = df[(df["run_id"] == run_id) & (df["problem_id"].str.startswith(problem_prefix[:40]))]
        if len(match) == 0:
            display(HTML(f"<div style='color:red;'>No match in {MODELS[model_key]['name']} for '{problem_prefix}'</div>"))
            continue

        row = match.iloc[0]
        entry = match_trace(model_key, row["problem_id"], run_id)

        correct_str = '✅' if row.get('agent_correct') else '❌'
        display(Markdown(f"---\n### {MODELS[model_key]['name']} {correct_str}"))

        ctx = (
            f"epistemic_state={row.get('epistemic_state')} | "
            f"trajectory={row.get('trajectory_type')} | "
            f"entropy={row.get('entropy')} | "
            f"searches={row.get('num_searches')}"
        )
        flags = [f for f in ["is_context_poisoning", "is_performative_ignorance",
                             "is_utilization_failure"] if f in row and row[f]]
        if flags:
            ctx += f" | flags: {', '.join(flags)}"
        display(HTML(f"<div style='font-size:12px; color:#555;'>{ctx}</div>"))

        if entry:
            render_trace(entry)
        else:
            display(HTML("<div style='color:orange;'>⚠️ Trace not found</div>"))

In [21]:
# Find questions that appear in both models for comparison
g_problems = set(analysis["gemini"][analysis["gemini"]["run_id"] == 1]["problem_id"])
n_problems = set(analysis["nemotron"][analysis["nemotron"]["run_id"] == 1]["problem_id"])
shared = g_problems & n_problems
print(f"Shared problems: {len(shared)}")

# Show a few interesting shared problems with different outcomes
g_df = analysis["gemini"][analysis["gemini"]["run_id"] == 1].set_index("problem_id")
n_df = analysis["nemotron"][analysis["nemotron"]["run_id"] == 1].set_index("problem_id")

interesting = []
for pid in shared:
    if pid in g_df.index and pid in n_df.index:
        g_row = g_df.loc[pid]
        n_row = n_df.loc[pid]
        g_traj = g_row.get("trajectory_type", "")
        n_traj = n_row.get("trajectory_type", "")
        if isinstance(g_traj, pd.Series): g_traj = g_traj.iloc[0]
        if isinstance(n_traj, pd.Series): n_traj = n_traj.iloc[0]
        if g_traj != n_traj and pd.notna(g_traj) and pd.notna(n_traj):
            interesting.append((pid, g_traj, n_traj))

print(f"\nProblems with different trajectories: {len(interesting)}")
print("\nExamples:")
for pid, gt, nt in interesting[:8]:
    print(f"  {pid[:60]}  Gemini={gt}  Nemotron={nt}")

Shared problems: 177

Problems with different trajectories: 120

Examples:
  Which company is Karosa C 954 produced by?...  Gemini=VERIFY_AND_CONFIRM  Nemotron=NEUTRAL_TRAJECTORY
  Who is Mary Lucier married to?...  Gemini=EXPLORE_AND_FIND  Nemotron=PRODUCTIVE_SEARCH
  What music label is Fit for a King represented by?...  Gemini=VERIFY_AND_CONFIRM  Nemotron=EXPLORE_AND_FIND
  What music label is Mike Dred represented by?...  Gemini=VERIFY_AND_CONFIRM  Nemotron=EXPLORE_AND_FIND
  Who is Sahib Jamal married to?...  Gemini=EXPLORE_AND_FIND  Nemotron=NEUTRAL_TRAJECTORY
  Who is the author of The Seven Fabulous Wonders?...  Gemini=HEDGE_AND_RESOLVE  Nemotron=EXPLORE_AND_FIND
  Who is the author of Tom Cruise: All the World's a...  Gemini=EXPLORE_AND_FIND  Nemotron=PRODUCTIVE_SEARCH
  What music label is Some People Change represented...  Gemini=NEUTRAL_TRAJECTORY  Nemotron=EXPLORE_AND_FIND


In [22]:
# Compare a specific question across both models
# Pick one from the interesting list above, or type any question prefix
if interesting:
    compare_models(interesting[0][0][:45])

---
### Gemini 3 Pro ✅

---
### Nemotron 3 Nano ✅

---
## 7. Custom Exploration

Use these cells for ad-hoc queries.

In [23]:
# Example: find all questions where Gemini is wrong despite HIGH_CERTAINTY
_ = browse(
    model="gemini",
    epistemic_state="HIGH_CERTAINTY",
    correct=False,
    max_display=2,
)

### Gemini 3 Pro — state=HIGH_CERTAINTY, correct=False
**14 matching rows** (showing up to 2 traces from run 1)

,problem_id,agent_correct,epistemic_state,trajectory_type,num_searches,entropy,hypothesis_correct
27,Who is the author of Limbo?...,False,HIGH_CERTAINTY,STUBBORN_REJECTION,2.0,0.7219,NO
35,Who is the author of Introduction to Christianity?...,False,HIGH_CERTAINTY,VERIFY_AND_CONFIRM,1.0,0.0000,YES
52,What music label is I Could Fall in Love represent...,False,HIGH_CERTAINTY,MISLED_BY_SEARCH,1.0,0.0000,YES
55,What music label is Skin represented by?...,False,HIGH_CERTAINTY,STUBBORN_REJECTION,7.0,0.0000,NO
84,What music label is Good Morning Beautiful represe...,False,HIGH_CERTAINTY,STUBBORN_REJECTION,5.0,0.0000,NO
104,Who is Fatimah married to?...,False,HIGH_CERTAINTY,VERIFY_AND_CONFIRM,1.0,0.0000,YES
113,Who is James Connolly married to?...,False,HIGH_CERTAINTY,MISLED_BY_SEARCH,1.0,0.0000,YES
117,Who is Mary Ritter Beard married to?...,False,HIGH_CERTAINTY,NaN,0.6,0.0000,YES
163,Which company is Macchi M.12 produced by?...,False,HIGH_CERTAINTY,MISLED_BY_SEARCH,1.0,0.9710,YES
167,Which company is AR-15 produced by?...,False,HIGH_CERTAINTY,VERIFY_AND_CONFIRM,2.0,0.0000,YES


In [24]:
# Example: Nemotron EXPLORE_AND_FIND that still got wrong
_ = browse(
    model="nemotron",
    trajectory="EXPLORE_AND_FIND",
    correct=False,
    max_display=2,
)

### Nemotron 3 Nano — trajectory=EXPLORE_AND_FIND, correct=False
**20 matching rows** (showing up to 2 traces from run 1)

,problem_id,agent_correct,epistemic_state,trajectory_type,num_searches,entropy,hypothesis_correct
27,Who is the author of Limbo?...,False,IGNORANCE,EXPLORE_AND_FIND,2.0,1.9219,NaN
46,Who is the author of Long Time Dead?...,False,SEARCH_FIRST,EXPLORE_AND_FIND,1.0,1.5219,NaN
57,What music label is Chris Stamey represented by?...,False,SEARCH_FIRST,EXPLORE_AND_FIND,1.0,1.9219,NaN
59,What music label is The Age of Fear represented by...,False,SEARCH_FIRST,EXPLORE_AND_FIND,1.0,1.9219,NaN
64,What music label is Ghost Dance represented by?...,False,SEARCH_FIRST,EXPLORE_AND_FIND,3.0,1.3710,NaN
67,What music label is Mike Dred represented by?...,False,SEARCH_FIRST,EXPLORE_AND_FIND,1.0,2.3219,NaN
69,What music label is The Evil Powers of Rock 'n' Ro...,False,SEARCH_FIRST,EXPLORE_AND_FIND,1.0,1.9219,NaN
72,What music label is Good Hit represented by?...,False,IGNORANCE,EXPLORE_AND_FIND,10.0,1.3710,NaN
87,What music label is James Taylor represented by?...,False,SEARCH_FIRST,EXPLORE_AND_FIND,1.0,1.3710,NaN
93,What music label is Significant Other represented ...,False,SEARCH_FIRST,EXPLORE_AND_FIND,9.0,1.3710,NaN


In [25]:
# Quick reference for all valid filter values:
print("trajectory types:", df_all["trajectory_type"].dropna().unique().tolist())
print("epistemic states:", df_all["epistemic_state"].dropna().unique().tolist())
print("pathology flags: is_context_poisoning, is_performative_ignorance, is_utilization_failure, is_confirmation_bias")
print("models: 'gemini', 'nemotron'")
print("run_id: 1-5")

trajectory types: ['EXPLORE_AND_FIND', 'STUBBORN_REJECTION', 'VERIFY_AND_CONFIRM', 'PRODUCTIVE_SEARCH', 'NEUTRAL_TRAJECTORY', 'HEDGE_AND_RESOLVE', 'MISLED_BY_SEARCH', 'EXPLORE_AND_FAIL', 'VERIFY_AND_CORRECT']
epistemic states: ['SEARCH_FIRST', 'HIGH_CERTAINTY', 'AMBIGUITY', 'IGNORANCE']
pathology flags: is_context_poisoning, is_performative_ignorance, is_utilization_failure, is_confirmation_bias
models: 'gemini', 'nemotron'
run_id: 1-5
